In [1]:
%pip install python-dotenv langchain-groq langgraph
from langchain_core.messages import HumanMessage, ToolMessage
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os

load_dotenv()

if os.getenv("GROQ_API_KEY"):
    print("Groq API key is set.")
else:
    raise ValueError("Groq API key is not set.")
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.3-70b-versatile"
)
model
model.invoke("tell me what should be do if our gf is angrey").content

c:\Users\puroh\Desktop\langgraph-tutoril\.venv\Scripts\python.exe: No module named pip


Note: you may need to restart the kernel to use updated packages.
Groq API key is set.


'A very common and sensitive topic. If your girlfriend is angry, it\'s essential to handle the situation with care and empathy. Here are some steps you can follow:\n\n1. **Stay calm**: It\'s crucial to remain calm and composed, even if your girlfriend is upset. Avoid getting defensive or matching her level of anger, as this can escalate the situation.\n2. **Listen actively**: Allow her to express her feelings and concerns without interrupting. Listen attentively to what she\'s saying, and try to understand her perspective.\n3. **Acknowledge her emotions**: Validate her feelings by acknowledging that you understand she\'s upset. You can say something like, "I can see that you\'re really upset, and I\'m here to listen."\n4. **Apologize if necessary**: If you\'ve done something to upset her, apologize sincerely. Make sure your apology is heartfelt and specific, and that you take responsibility for your actions.\n5. **Give her space**: If she needs some time to cool down, give her space. R

In [2]:
from typing import Literal, Optional, TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt


class ApprovalState(TypedDict):
    action_details: str
    status: Optional[Literal["pending", "approved", "rejected"]]


def approval_node(state: ApprovalState) -> Command[Literal["proceed", "cancel"]]:
    # Expose details so the caller can render them in a UI
    decision = interrupt({
        "question": "Approve this action?",
        "details": state["action_details"],
    })

    # Route to the appropriate node after resume
    return Command(goto="proceed" if decision else "cancel")


def proceed_node(state: ApprovalState):
    return {"status": "approved"}


def cancel_node(state: ApprovalState):
    return {"status": "rejected"}


builder = StateGraph(ApprovalState)
builder.add_node("approval", approval_node)
builder.add_node("proceed", proceed_node)
builder.add_node("cancel", cancel_node)
builder.add_edge(START, "approval")
builder.add_edge("proceed", END)
builder.add_edge("cancel", END)

# Use a more durable checkpointer in production
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "approval-456"}}
initial = graph.invoke(
    {"action_details": "Send Daily Summary", "status": "pending"},
    config=config,
)
print(initial["__interrupt__"])  # -> [Interrupt(value={'question': ..., 'details': ...})]

[Interrupt(value={'question': 'Approve this action?', 'details': 'Send Daily Summary'}, id='67babf81b2f68e934fbc0d4e4b39d7fc')]
